# Loading and Overview

In [4]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Import data with only necessary columns
columns_needed = ['isBounty', 'initialExperienceReceiving', 'initialExperienceGiving',
                 'timeSinceFirstActivityDays', 'userId', 'numHelpProvidedAT', 'numQuestionsAskedAT', 'questionId']

df = pd.read_parquet('../data/study_datasets/user_answers_bounty_processed.parquet',
                    columns=columns_needed)

df['initialExperienceReceiving'] = pd.Categorical(df['initialExperienceReceiving'],
                                                 categories=['no help seeked', 'help seeked', 'help received'])

df['initialExperienceGiving'] = pd.Categorical(df['initialExperienceGiving'],
                                              categories=['no help attempted', 'help attempted', 'helped'])

print(f"\nColumn types:")
print(df.dtypes)

print("\n\n==== OVERVIEW TABLE ====")
print("\nCounts by isBounty and initialExperienceReceiving:")
receiving_crosstab = pd.crosstab(df['isBounty'], df['initialExperienceReceiving'],
                                margins=True, margins_name="Total")
print(receiving_crosstab)

# Calculate percentages for isBounty=1 across initialExperienceReceiving
print("\nPercentages when isBounty=1 by initialExperienceReceiving:")
bounty_receiving = receiving_crosstab.loc[1, :]  # Get row where isBounty=1
bounty_receiving_pct = (bounty_receiving / bounty_receiving['Total'] * 100).round(2)
print(bounty_receiving_pct)

print("\nCounts by isBounty and initialExperienceGiving:")
giving_crosstab = pd.crosstab(df['isBounty'], df['initialExperienceGiving'],
                             margins=True, margins_name="Total")
print(giving_crosstab)

# Calculate percentages for isBounty=1 across initialExperienceGiving
print("\nPercentages when isBounty=1 by initialExperienceGiving:")
bounty_giving = giving_crosstab.loc[1, :]  # Get row where isBounty=1
bounty_giving_pct = (bounty_giving / bounty_giving['Total'] * 100).round(2)
print(bounty_giving_pct)


Column types:
isBounty                         int64
initialExperienceReceiving    category
initialExperienceGiving       category
timeSinceFirstActivityDays     float64
userId                           int64
numHelpProvidedAT                int64
numQuestionsAskedAT              int64
questionId                       int64
dtype: object


==== OVERVIEW TABLE ====

Counts by isBounty and initialExperienceReceiving:
initialExperienceReceiving  no help seeked  help seeked  help received  \
isBounty                                                                 
0                                  9603368      3179827       19726911   
1                                   104096        40901         201155   
Total                              9707464      3220728       19928066   

initialExperienceReceiving     Total  
isBounty                              
0                           32510106  
1                             346152  
Total                       32856258  

Percentages w

# No FE Models: Experience Giving / Receiving

In [24]:
from statsmodels.formula.api import logit
# Define formulas for models without fixed effects
formula1 = 'isBounty ~ C(initialExperienceReceiving) + timeSinceFirstActivityDays'
formula2 = 'isBounty ~ C(initialExperienceGiving) + timeSinceFirstActivityDays'

# Model names for without FE
model_names_no_fe = ["Receiving Experience Model", "Giving Experience Model"]
formulas_no_fe = [formula1, formula2]
filtered_df = df[(df['numQuestionsAskedAT'].isin([0, 1])) & (df['numHelpProvidedAT'] == 0)].reset_index(drop=True)

# Run and display each model individually (without FE)
models_no_fe = []
for i, (formula, name) in enumerate(zip(formulas_no_fe, model_names_no_fe)):
    # Fit the model
    model = smf.logit(formula=formula, data=filtered_df).fit(cov_type='cluster', cov_kwds={'groups': filtered_df[['questionId', 'userId']].apply(tuple, axis=1)})

    models_no_fe.append(model)

    # Create a Stargazer table for this single model
    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Initial Experience on Bounty Usage - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    # Display HTML output for this model
    html_output = single_stargazer.render_html()
    display(HTML(html_output))

Optimization terminated successfully.
         Current function value: 0.033861
         Iterations 9


Optimization terminated successfully.
         Current function value: 0.033938
         Iterations 9


LinAlgError: Singular matrix

# Rare Events Regression (No FE)

# User FE: Experience Giving / Receiving

In [15]:
# Define formulas for models with demeaned dependent variable
formula3 = 'userFeIsBounty ~ C(initialExperienceReceiving) + timeSinceFirstActivityDays'
formula4 = 'userFeIsBounty ~ C(initialExperienceGiving) + timeSinceFirstActivityDays'

# Model names for demeaned models
model_names_demeaned = ["Demeaned Receiving Experience Model", "Demeaned Giving Experience Model"]
formulas_demeaned = [formula3, formula4]
filtered_df = df[(df['numQuestionsAskedAT'].isin([0, 1])) & (df['numHelpProvidedAT'] == 0)]

# Calculate user-level means
user_means = filtered_df.groupby('userId')['isBounty'].mean().reset_index()
user_means.columns = ['userId', 'userMeanIsBounty']
filtered_df = filtered_df.merge(user_means, on='userId', how='left')
filtered_df['userFeIsBounty'] = filtered_df['isBounty'] - filtered_df['userMeanIsBounty']

# Run and display each demeaned model individually
models_demeaned = []
for i, (formula, name) in enumerate(zip(formulas_demeaned, model_names_demeaned)):
    # Fit the model
    model = smf.ols(formula=formula, data=filtered_df).fit()

    # Apply clustered standard errors by userId
    model = model.get_robustcov_results(
        cov_type='cluster',
        groups=filtered_df['userId']
    )

    models_demeaned.append(model)

    # Create a Stargazer table for this single model
    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Initial Experience on Bounty Usage (User FE) - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    # Display HTML output for this model
    html_output = single_stargazer.render_html()
    display(HTML(html_output))

C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 0
  warnings.warn('covariance of constraints does not have full '
C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1923: RuntimeWarning: invalid value encountered in divide
  F /= J


C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 0
  warnings.warn('covariance of constraints does not have full '
C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1923: RuntimeWarning: invalid value encountered in divide
  F /= J


# FE for all Users

In [17]:
# Define formulas for models with demeaned dependent variable
formula3 = 'userFeIsBounty ~ C(initialExperienceReceiving) + timeSinceFirstActivityDays'
formula4 = 'userFeIsBounty ~ C(initialExperienceGiving) + timeSinceFirstActivityDays'

# Model names for demeaned models
model_names_demeaned = ["Demeaned Receiving Experience Model", "Demeaned Giving Experience Model"]
formulas_demeaned = [formula3, formula4]

# Calculate user-level means
user_means = df.groupby('userId')['isBounty'].mean().reset_index()
user_means.columns = ['userId', 'userMeanIsBounty']
df = df.merge(user_means, on='userId', how='left')
df['userFeIsBounty'] = df['isBounty'] - df['userMeanIsBounty']

# Run and display each demeaned model individually
models_demeaned = []
for i, (formula, name) in enumerate(zip(formulas_demeaned, model_names_demeaned)):
    # Fit the model
    model = smf.ols(formula=formula, data=df).fit()

    # Apply clustered standard errors by userId
    model = model.get_robustcov_results(
        cov_type='cluster',
        groups=df['userId']
    )

    models_demeaned.append(model)

    # Create a Stargazer table for this single model
    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Initial Experience on Bounty Usage (User FE) - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    # Display HTML output for this model
    html_output = single_stargazer.render_html()
    display(HTML(html_output))